# Apt 305 — effect of each ISO 52016-1 engine change

Runs the same Melbourne apartment through three engine versions and reports the effect of each change as a table and a bar chart.

| Branch | Engine |
|---|---|
| `claude/pybuildingenergy-baseline-anjro8` | Unmodified ISO 52016-1 |
| `claude/dynamic-window-properties-anjro8` | + dynamic window properties (`U_win(t)`, angular `g_win`) |
| `claude/window-plus-dynamic-hce-anjro8` | + wind-dependent surface heat transfer coefficients |

Each branch runs in its own subprocess — three versions of the same `pybuildingenergy` package cannot share one `sys.path`.

**Runtime:** roughly 5–10 minutes end to end (dependency install dominates).

> **Private repo?** If the clone below asks for credentials, create a GitHub personal access token with `repo` scope and use the commented-out line instead.

In [ ]:
# 1. Clone the repository (all branches — the comparison needs them)
REPO = "https://github.com/samiraghafarigousheh-sys/AIB.git"
BRANCH = "claude/window-plus-dynamic-hce-anjro8"

# Private repo? Uncomment and paste a token with `repo` scope:
# TOKEN = "ghp_xxx"
# REPO = f"https://{TOKEN}@github.com/samiraghafarigousheh-sys/AIB.git"

import os, shutil
if os.path.isdir("AIB"):
    shutil.rmtree("AIB")

!git clone --quiet --branch $BRANCH $REPO AIB
%cd AIB

# Make sure every branch is present locally, so `git worktree` can reach them.
!git fetch --quiet origin '+refs/heads/*:refs/remotes/origin/*'
!git branch -r

In [ ]:
# 2. Dependencies
!pip install -q -r pybuildingenergy/requirements.txt

# git needs an identity before `worktree add` will run in a fresh container
!git config user.email "colab@example.com"
!git config user.name  "Colab"
print("dependencies installed")

In [ ]:
# 3. Melbourne weather
#
# 'auto' tries, in order: a cached Melbourne EPW, a download from public TMY
# mirrors, then PVGIS at the building's own coordinates (-37.800, 144.968).
#
# It will NOT quietly substitute another city: with weather_source='epw' the
# engine reads latitude from the EPW header rather than from the building
# dictionary, so a stand-in file would silently relocate the apartment.
import sys
sys.path.insert(0, "examples")
from weather_melbourne import resolve, WeatherUnavailable

try:
    source, path, label = resolve(None, "auto")
    print(f"weather source : {source}")
    print(f"weather        : {label}")
except WeatherUnavailable as exc:
    print(exc)

In [ ]:
# 4. Run all three branches (this is the slow cell — three annual simulations)
!python examples/compare_branches_apt305.py --outdir results/apt305

In [ ]:
# 5. Table
import pandas as pd
df = pd.read_csv("results/apt305/comparison.csv")
display(df)

In [ ]:
# 6. Bar chart
from IPython.display import Image, display
display(Image("results/apt305/apt305_comparison.png"))

## Varying the building

`examples/apt305_building.py` holds the building dictionary and imports no engine, so the same definition feeds every engine version. Edit it and re-run cell 4 to test a different fabric.

## Using your own weather file

```python
!python examples/compare_branches_apt305.py --weather /content/AUS_VIC_Melbourne.epw
```

The file is validated against the building's coordinates and rejected if it is from elsewhere. To override that deliberately, add `--allow-site-mismatch` — results are then for the EPW's own location, not Melbourne, and are labelled that way on the chart.

## Isolating a single change without switching branches

Both changes are on by default on the top branch and can be switched off individually:

| Option | Effect |
|---|---|
| `dynamic_window_properties=False` | Recovers exact baseline behaviour |
| `dynamic_surface_heat_transfer=False` | Recovers change-1-only behaviour |
| `window_angular_solar_model='none'` | Disables the Karlsson–Roos angular correction |

Both are verified inert when disabled: every compared metric matches the reference engine exactly, at zero tolerance.